# Student Performance — Notebook 3: My Additions (EDA)
**Project:** End-to-End AI Framework for Student Performance Analysis
**Student:** Megha Deopa | PRN: 2405020011520 | MBA AI/ML July 2024

**What is new in this notebook (not in Notebook 1):**
- Correlation Heatmap
- Pass/Fail Analysis
- Grade Bucketing (A/B/C/D/F)
- Subject-wise comparison
- Extended Gender analysis (all 3 subjects)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

# Load data and recreate engineered columns
df = pd.read_csv('data/stud.csv')
df['total score'] = df['math_score'] + df['reading_score'] + df['writing_score']
df['average']     = df['total score'] / 3

print('Dataset loaded! Shape:', df.shape)

## Addition 1: Correlation Heatmap
**Why:** Mentor used pairplot but never created a heatmap with actual correlation values

In [ ]:
plt.figure(figsize=(9, 7))
corr = df[['math_score','reading_score','writing_score','total score','average']].corr()

sns.heatmap(corr,
            annot=True,        # Show numbers inside
            fmt='.2f',         # 2 decimal places
            cmap='coolwarm',   # Red = high, Blue = low
            linewidths=0.5,
            square=True,
            vmin=0, vmax=1)

plt.title('Correlation Heatmap — Student Scores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey Correlation Values:')
print(f'Reading vs Writing : {corr["reading_score"]["writing_score"]:.2f} — Very Strong')
print(f'Math vs Reading    : {corr["math_score"]["reading_score"]:.2f} — Strong')
print(f'Math vs Writing    : {corr["math_score"]["writing_score"]:.2f} — Strong')

## Addition 2: Pass/Fail Column and Analysis
**Why:** Mentor ONLY did regression (predict number). I add classification target (Pass/Fail)

In [ ]:
# Create pass/fail column (threshold: average >= 40 = Pass)
df['pass_fail'] = df['average'].apply(lambda x: 'Pass' if x >= 40 else 'Fail')

pass_count = df['pass_fail'].value_counts()
print('Pass/Fail Distribution:')
print(pass_count)
print(f'\nPass Rate: {pass_count["Pass"]/len(df)*100:.1f}%')
print(f'Fail Rate: {pass_count["Fail"]/len(df)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall pass/fail
sns.countplot(x='pass_fail', data=df,
              palette={'Pass':'#2ecc71','Fail':'#e74c3c'}, ax=axes[0])
axes[0].set_title('Overall Pass vs Fail Distribution', fontsize=13)
axes[0].set_xlabel('Result')
axes[0].set_ylabel('Number of Students')
for container in axes[0].containers:
    axes[0].bar_label(container, fontsize=13)

# By gender
sns.countplot(x='gender', hue='pass_fail', data=df,
              palette={'Pass':'#2ecc71','Fail':'#e74c3c'}, ax=axes[1])
axes[1].set_title('Pass vs Fail by Gender', fontsize=13)
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Number of Students')
for container in axes[1].containers:
    axes[1].bar_label(container, fontsize=12)

plt.tight_layout()
plt.show()

## Addition 3: Grade Bucketing (A / B / C / D / F)
**Why:** Raw scores alone are not intuitive. Grade labels make insights actionable for educators

In [ ]:
def assign_grade(score):
    if   score >= 90: return 'A'
    elif score >= 75: return 'B'
    elif score >= 60: return 'C'
    elif score >= 40: return 'D'
    else:             return 'F'

df['grade'] = df['average'].apply(assign_grade)

print('Grade Distribution:')
grade_counts = df['grade'].value_counts().sort_index()
for grade, count in grade_counts.items():
    print(f'  Grade {grade}: {count} students ({count/len(df)*100:.1f}%)')

In [ ]:
order  = ['A','B','C','D','F']
colors = ['#27ae60','#2980b9','#f39c12','#e67e22','#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall grade distribution
sns.countplot(x='grade', data=df, order=order, palette=dict(zip(order,colors)), ax=axes[0])
axes[0].set_title('Overall Grade Distribution', fontsize=13)
for container in axes[0].containers:
    axes[0].bar_label(container, fontsize=12)

# Grade by lunch type
sns.countplot(x='grade', hue='lunch', data=df, order=order, palette='Set1', ax=axes[1])
axes[1].set_title('Grade Distribution by Lunch Type', fontsize=13)

plt.suptitle('Student Grade Distribution (A to F)', fontsize=15)
plt.tight_layout()
plt.show()

## Addition 4: Subject-wise Performance Comparison
**Why:** Mentor never compared all 3 subjects side by side in a single clean chart

In [ ]:
subjects = ['Math', 'Reading', 'Writing']
means    = [df['math_score'].mean(), df['reading_score'].mean(), df['writing_score'].mean()]
stds     = [df['math_score'].std(),  df['reading_score'].std(),  df['writing_score'].std()]
mins     = [df['math_score'].min(),  df['reading_score'].min(),  df['writing_score'].min()]
maxs     = [df['math_score'].max(),  df['reading_score'].max(),  df['writing_score'].max()]

summary = pd.DataFrame({'Subject':subjects,'Mean':means,'Std Dev':stds,'Min':mins,'Max':maxs})
print(summary.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(subjects, means, color=['#e74c3c','#3498db','#2ecc71'], width=0.4, zorder=3)

for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.8,
            f'Mean: {mean:.1f}\nSD: {std:.1f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Average Score Across All Three Subjects', fontsize=14)
ax.set_ylabel('Average Score')
ax.set_ylim(0, 84)
ax.grid(axis='y', alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

## Addition 5: Gender Deep Dive — All 3 Subjects
**Why:** Mentor compared only math and total average. I compare all three subjects

In [ ]:
gender_scores = df.groupby('gender')[['math_score','reading_score','writing_score']].mean()
print('Average scores by gender:')
print(gender_scores.round(2))

fig, ax = plt.subplots(figsize=(10, 6))
x     = np.arange(3)
width = 0.3

b1 = ax.bar(x - width/2, gender_scores.loc['male'],   width, label='Male',   color='#3498db', alpha=0.85)
b2 = ax.bar(x + width/2, gender_scores.loc['female'], width, label='Female', color='#e91e63', alpha=0.85)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f'{bar.get_height():.1f}',
            ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(['Math Score','Reading Score','Writing Score'])
ax.set_ylabel('Average Score')
ax.set_title('Gender vs Performance in All Three Subjects', fontsize=14)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 80)
plt.tight_layout()
plt.show()

m = gender_scores.loc['male']
f = gender_scores.loc['female']
print(f'\nMales lead Math by     : {m["math_score"]    - f["math_score"]:.1f} pts')
print(f'Females lead Reading by: {f["reading_score"] - m["reading_score"]:.1f} pts')
print(f'Females lead Writing by: {f["writing_score"] - m["writing_score"]:.1f} pts')